In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../../")
sys.path.append('../../src')

%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import random
import matplotlib.pyplot as plt

from m3util.ml.rand import set_seeds
from m3util.viz.style import set_style
from m3util.viz.printing import printer
from belearn.viz.viz import Viz
from belearn.dataset.dataset import BE_Dataset
from belearn.functions.sho import SHO_nn

from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model
from autophyslearn.postprocessing.complex import ComplexPostProcessor

from m3util.viz.layout import inset_connector, add_box
from m3util.viz.text import set_sci_notation_label, labelfigs, bring_text_to_front

from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg

printing = printer(basepath = './Figures/')


set_style("printing")
set_seeds(seed=42)

%matplotlib inline

In [ ]:
def SHO_fit_func_nn(params,
                    wvec_freq,
                    device='cpu'):
    """_summary_

    Returns:
        _type_: _description_
    """

    Amp = params[:, 0].type(torch.complex128)
    w_0 = params[:, 1].type(torch.complex128)
    Q = params[:, 2].type(torch.complex128)
    phi = params[:, 3].type(torch.complex128)
    wvec_freq = torch.tensor(wvec_freq)

    Amp = torch.unsqueeze(Amp, 1)
    w_0 = torch.unsqueeze(w_0, 1)
    phi = torch.unsqueeze(phi, 1)
    Q = torch.unsqueeze(Q, 1)

    wvec_freq = wvec_freq.to(device)

    numer = Amp * torch.exp((1.j) * phi) * torch.square(w_0)
    den_1 = torch.square(wvec_freq)
    den_2 = (1.j) * wvec_freq.to(device) * w_0 / Q
    den_3 = torch.square(w_0)

    den = den_1 - den_2 - den_3

    func = numer / den

    return func

In [ ]:
# Specify the filename and the path to save the file
filename = "data_raw.h5"
save_path = "./Data"

data_path = save_path + "/" + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# print the contents of the file
dataset.print_be_tree()

In [ ]:
#dataset.SHO_Scaler()

In [ ]:
# h5_loop_fit, h5_loop_group = dataset.LSQF_Loop_Fit()

In [ ]:
%load_ext autoreload
%autoreload 2

from belearn.dataset.dataset import BE_Dataset
from belearn.viz.viz import Viz
from m3util.viz.printing import printer
printing = printer(basepath = './Figures/')

In [ ]:
# instantiate the visualization object
image_scalebar = [2000, 500, "nm", "br"]

In [ ]:

# BE_viz = Viz(dataset, printing, verbose=True, 
#              SHO_ranges = [(0,1.5e-4), (1.31e6, 1.33e6), (-300, 300), (-np.pi, np.pi)], 
#              image_scalebar = image_scalebar)

# instantiates the visualization object
BE_viz = Viz(dataset, printing, verbose=True)

In [ ]:

from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model
from autophyslearn.postprocessing.complex import ComplexPostProcessor

In [ ]:
#from m3_learning.be.loop_fitter import loop_fitting_function_torch
from m3util.ml.optimizers.TrustRegion import TRCG
import torch.optim as optim
from belearn.functions.hysteresis import hysteresis_nn


datafed_path = "2024_SHO_Fitting/Original_NN_SHO_Fitter"
device = 'cuda:1'

data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
# V = np.swapaxes(np.atleast_2d(dataset.get_voltage), 0, 1).astype(np.float64)


model_ = Multiscale1DFitter(
            #BE_viz.loop_fitting_function_torch, # function 
                hysteresis_nn,  # function

                            voltage[:,0].squeeze(), # x data
#                             V.squeeze(),
                            1, # input channels
                            9, # output parameters
                            dataset.loop_param_scaler,
                            loops_scaler=dataset.hysteresis_scaler,
                            device=device
                            )

# instantiate the model
model = Model(model_, dataset, training=True, model_basename="SHO_Fitter_original_data",
                datafed_path=datafed_path,
                script_path="/home/jca92/Rapid-Fitting-BEPFM-NN/notebooks/7_Hysteresis_Fitter.ipynb",
                device=device)

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(data.reshape(-1,96), test_size=0.2, random_state=42, shuffle=True)

X_train = np.atleast_3d(X_train)

optimizer = {
    "name": "TRCG", 
    "optimizer": TRCG,
    "closure_size": 1,
    "cgopttol": 1e-3,
    "c0tr": 0.2,
    "c1tr": 0.25,
    "c2tr": 0.75,
    "t1tr": 0.75,
    "t2tr": 2.0,
    "radius_max": 5.0,  
    "radius_initial": 1.0,
    "radius" : 1.0,
    "device": device,
    "ADAM_epochs": 100}


train =  True

if train:
    # fits the model
    model.fit(
        X_train,
        1024,
        optimizer=optimizer,
        epochs = 500,
    )
else:
    model.load(
       # "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_epoch_5_train_loss_0.0449272525189978.pth"
       "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_499_train_loss_0.00602861393450035.pth"
    )